# 02 — Baseline Attribution Graphs (base model)

Runs attribution graph generation on **base Gemma-2-2B** (no fine-tuning)
over the **analysis split** (75 prompts).  Computes structural statistics
for each prompt and saves them to `results/statistics/stats_base.json`.

**Run as a script** to avoid kernel timeouts:
```bash
jupyter nbconvert --to script 02_baseline_attribution.ipynb
python 02_baseline_attribution.py
```

Saves incrementally (every prompt) so a restart resumes from the last checkpoint.

**Prerequisites:** `01_dataset_generation.ipynb` must have been run.

## 0 — Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
else:
    print("WARNING: could not locate circuit_tracer repo.")

try:
    from dotenv import load_dotenv
    if _root is not None and (_root / ".env").is_file():
        load_dotenv(_root / ".env")
        print("Loaded .env")
except ImportError:
    pass

# Persistent HF cache for RunPod
if IN_RUNPOD:
    persistent_root = Path(os.environ.get("RUNPOD_PERSISTENT_ROOT", "/workspace")).resolve()
    hf_home = persistent_root / "hf"
    for key, path in {
        "HF_HOME": hf_home,
        "HUGGINGFACE_HUB_CACHE": hf_home / "hub",
        "TRANSFORMERS_CACHE": hf_home / "transformers",
    }.items():
        path.mkdir(parents=True, exist_ok=True)
        os.environ[key] = str(path)
    print(f"RunPod cache: {hf_home}")

MY_WORK = _my_work if _root else Path(".").resolve()
print(f"MY_WORK: {MY_WORK}")

## 1 — Imports & constants

In [ ]:
import json
import time
import traceback

import torch
import numpy as np

# ── Plan constants (never change) ──────────────────────────────────────────────
MODEL_NAME      = "google/gemma-2-2b"
TRANSCODER_NAME = "gemma"
TOKEN_TRUE      = " True"
TOKEN_FALSE     = " False"
VOCAB_ID_TRUE   = 5569
VOCAB_ID_FALSE  = 7662

ATTR_KWARGS = dict(
    batch_size=256,
    max_feature_nodes=8192,
    offload="disk",
    verbose=False,
)

PHASE = "base"

# ── Paths ──────────────────────────────────────────────────────────────────────
ANALYSIS_FILE  = MY_WORK / "data" / "splits" / "analysis.jsonl"
GRAPHS_DIR     = MY_WORK / "results" / "graphs_base"
STATS_FILE     = MY_WORK / "results" / "statistics" / "stats_base.json"

GRAPHS_DIR.mkdir(parents=True, exist_ok=True)
STATS_FILE.parent.mkdir(parents=True, exist_ok=True)

print(f"Analysis file : {ANALYSIS_FILE}")
print(f"Graphs dir    : {GRAPHS_DIR}")
print(f"Stats output  : {STATS_FILE}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device        : {device}")

## 2 — Load analysis split

In [ ]:
analysis_prompts = []
with open(ANALYSIS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        analysis_prompts.append(json.loads(line.strip()))

print(f"Loaded {len(analysis_prompts)} analysis prompts")
print(f"  True : {sum(1 for p in analysis_prompts if p['label'])}")
print(f"  False: {sum(1 for p in analysis_prompts if not p['label'])}")

## 3 — Load model

In [ ]:
from circuit_tracer import ReplacementModel

model = ReplacementModel.from_pretrained(
    MODEL_NAME,
    TRANSCODER_NAME,
    dtype=torch.bfloat16,
    backend="transformerlens",
    device=device,
    lazy_encoder=True,
    lazy_decoder=True,
)
tokenizer = model.tokenizer

# Verify token IDs match constants
id_true  = tokenizer.encode(TOKEN_TRUE,  add_special_tokens=False)[-1]
id_false = tokenizer.encode(TOKEN_FALSE, add_special_tokens=False)[-1]
assert id_true  == VOCAB_ID_TRUE,  f"TOKEN_TRUE id mismatch: {id_true} != {VOCAB_ID_TRUE}"
assert id_false == VOCAB_ID_FALSE, f"TOKEN_FALSE id mismatch: {id_false} != {VOCAB_ID_FALSE}"
print(f"Token IDs verified: True={id_true}, False={id_false}")

## 4 — First-token accuracy check (pre-attribution)

Quick sanity check: does the base model's top-1 token match the label?
Expected ~50–65%.  If >80%, note it — it constrains fine-tuning interpretability.

In [ ]:
correct = 0
correct_true = 0
total_true = 0
correct_false = 0
total_false = 0

with torch.inference_mode():
    for entry in analysis_prompts:
        inputs = tokenizer(entry["prompt"], return_tensors="pt").to(device)
        logits = model.model(inputs.input_ids)[0]  # (1, seq_len, vocab)
        last_logit = logits[0, -1, :]
        pred_id = int(last_logit.argmax().item())
        label_id = VOCAB_ID_TRUE if entry["label"] else VOCAB_ID_FALSE
        if pred_id == label_id:
            correct += 1
        if entry["label"]:
            total_true += 1
            if pred_id == VOCAB_ID_TRUE:
                correct_true += 1
        else:
            total_false += 1
            if pred_id == VOCAB_ID_FALSE:
                correct_false += 1

n = len(analysis_prompts)
print(f"Base model first-token accuracy on analysis split:")
print(f"  Overall : {correct}/{n} = {correct/n:.1%}")
print(f"  True    : {correct_true}/{total_true} = {correct_true/total_true:.1%}")
print(f"  False   : {correct_false}/{total_false} = {correct_false/total_false:.1%}")

if correct / n > 0.80:
    print()
    print("NOTE: Base accuracy >80%. Fine-tuning may show limited circuit shift.")

## 5 — Attribution loop

Runs attribution for each analysis prompt.  Saves incrementally.

- Already-computed prompt IDs are skipped (resume support).
- Graph files are saved per-prompt under `results/graphs_base/`.
- Statistics are appended to `stats_base.json` after every prompt.
- Minimum success rate: 70% required to proceed to Stage 2.

In [ ]:
from circuit_tracer import attribute
from circuit_tracer.graph import create_graph_files

from utils.graph_statistics import (
    compute_statistics,
    load_statistics,
    append_statistic,
)

# ── Resume: load already-computed entries ──────────────────────────────────────
done_ids: set[str] = set()
if STATS_FILE.exists():
    existing = load_statistics(STATS_FILE)
    done_ids = {e["prompt_id"] for e in existing}
    print(f"Resuming: {len(done_ids)} prompts already done.")

remaining = [p for p in analysis_prompts if p["prompt_id"] not in done_ids]
print(f"Remaining: {len(remaining)} prompts")

# ── Attribution loop ───────────────────────────────────────────────────────────
t0 = time.time()
n_success = 0
n_fail = 0

for i, entry in enumerate(remaining):
    pid = entry["prompt_id"]
    print(f"[{i+1}/{len(remaining)}] {pid} ...", end=" ", flush=True)

    try:
        graph = attribute(
            prompt=entry["prompt"],
            model=model,
            attribution_targets=[TOKEN_TRUE, TOKEN_FALSE],
            **ATTR_KWARGS,
        )

        # Save graph files
        graph_out = GRAPHS_DIR / pid
        graph_out.mkdir(parents=True, exist_ok=True)
        create_graph_files(graph, str(graph_out))

        # Compute statistics
        stat = compute_statistics(graph, entry, phase=PHASE)
        n_success += 1
        print(f"OK  n_active={stat['n_active_features']}  density={stat['edge_density']:.4f}")

    except Exception as exc:
        stat = {
            "prompt_id": pid,
            "phase": PHASE,
            "label": entry["label"],
            "label_token": entry["label_token"],
            "template_id": entry["template_id"],
            "attribution_succeeded": False,
            "_error": str(exc),
        }
        n_fail += 1
        print(f"FAIL {exc}")

    append_statistic(stat, STATS_FILE)

elapsed = time.time() - t0
n_total = len(analysis_prompts)
n_done  = n_success + n_fail + len(done_ids)
success_rate = n_success / max(n_total - len(done_ids), 1)

print()
print(f"Done. Elapsed: {elapsed:.0f}s")
print(f"  New successes : {n_success}")
print(f"  New failures  : {n_fail}")
print(f"  Session success rate: {success_rate:.1%}")

## 6 — Summary and go/no-go check

In [ ]:
from utils.graph_statistics import load_statistics, aggregate_statistics
import json

all_stats = load_statistics(STATS_FILE)
agg = aggregate_statistics(all_stats)

print(f"Total prompts in stats file : {agg['n_total']}")
print(f"Successfully attributed     : {agg['n_succeeded']}")
print(f"Success rate                : {agg['success_rate']:.1%}")
print()

metrics = ['n_active_features','edge_density','mean_top50_score',
           'top10_over_top50','layer_entropy','mean_error_node_weight']
print(f"{'Metric':<28} {'Mean':>10} {'Std':>10} {'Median':>10} {'IQR':>10}")
print("-" * 72)
for m in metrics:
    v = agg.get(m)
    if v:
        print(f"{m:<28} {v['mean']:>10.4f} {v['std']:>10.4f} {v['median']:>10.4f} {v['iqr']:>10.4f}")
    else:
        print(f"{m:<28} {'N/A':>10}")

print()
sr = agg['success_rate']
if sr < 0.70:
    print("STOP: Success rate below 70%. Investigate prompt set before proceeding.")
elif sr < 0.85:
    print(f"NOTE: Success rate {sr:.1%} is below 85%. Proceed but flag as limitation.")
else:
    print(f"Success rate {sr:.1%} — OK to proceed to notebook 03.")